In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from ast import literal_eval
data = pd.read_csv("/Users/rohan/Documents/Research/forISI.csv")


In [ ]:
data.columns

In [ ]:
type(data['ISIString'].iloc[0])

In [ ]:
data

In [ ]:
import ast

# Assuming data is a pandas DataFrame and 'ISIString' is a column containing strings
isi_string = data['ISIString']





In [ ]:

isi_string= isi_string[:].replace(',',' ')
data['ISIString']=isi_string

In [ ]:
import pandas as pd

# Load the ISIString column from the CSV
data = pd.read_csv("/Users/rohan/Documents/Research/forISI.csv", usecols=['ISIString'])

# Function to remove commas, semicolons, and convert string to a list of floats
def process_isi_string(isi_string):
    try:
        # Replace commas and semicolons with spaces
        clean_string = isi_string.replace(',', ' ').replace(';', ' ')
        # Split the cleaned string into parts and convert to float
        return [float(x) for x in clean_string.split()]
    except ValueError as e:
        # Print or log the error along with the problematic string
        print(f"Could not convert string to float: '{isi_string}' - Error: {e}")
        return None  # Or use an empty list [] as the return value, depending on your needs

# Apply the function to the ISIString column
data['ProcessedISI'] = data['ISIString'].apply(process_isi_string)

# Optionally, you might want to remove or investigate rows where conversion failed
data = data[data['ProcessedISI'].notna()]

# Display the first few rows of the modified DataFrame to verify changes
print(data.head())


In [ ]:
#Below code to plot the histogram of the ISI values for WT and HET neurons separately 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load necessary columns from the CSV
data = pd.read_csv("/Users/rohan/Documents/Research/forISI.csv", usecols=['DIV', 'NeuronType', 'ISIString'])

# Filter data for DIV == 20
data = data[data['DIV'] == 20]

# Function to remove commas, semicolons, and convert string to a list of floats
def process_isi_string(isi_string):
    try:
        # Replace commas and semicolons with spaces
        clean_string = isi_string.replace(',', ' ').replace(';', ' ')
        # Split the cleaned string into parts and convert to float
        return [float(x) for x in clean_string.split()]
    except ValueError as e:
        print(f"Could not convert string to float: '{isi_string}' - Error: {e}")
        return None

# Apply the function to the ISIString column
data['ProcessedISI'] = data['ISIString'].apply(process_isi_string)

# Remove rows where conversion failed
data = data[data['ProcessedISI'].notna()]

# Calculate instantaneous frequencies (1/ISI)
data['InstantFreq'] = data['ProcessedISI'].apply(lambda x: [1/i for i in x if i != 0])

# Define bins for histogram
bins = np.linspace(0, 50, 101)  # Define bin edges as needed

# Function to compute histogram for each row
def compute_histogram(instant_freq):
    hist, _ = np.histogram(instant_freq, bins=bins)
    return hist

# Compute histograms for each row
data['Histogram'] = data['InstantFreq'].apply(compute_histogram)

# Aggregate histograms by neuron type
grouped_histograms = data.groupby('NeuronType')['Histogram'].agg(list)

# Calculate mean and standard deviation of histograms
results = {}
for neuron_type, histograms in grouped_histograms.items():
    # Stack histograms into a 2D array
    histograms_stack = np.vstack(histograms)
    mean_hist = np.mean(histograms_stack, axis=0)
    std_hist = np.std(histograms_stack, axis=0)
    results[neuron_type] = (mean_hist, std_hist)

    # Plotting
    plt.figure(figsize=(10, 5))
    plt.bar(bins[:-1], mean_hist, yerr=std_hist, width=np.diff(bins)[0], alpha=0.5, label=f'{neuron_type} Neurons')
    plt.xlabel('Instantaneous Frequency (Hz)')
    plt.ylabel('Frequency Count')
    plt.title(f'Average Histogram of Instantaneous Frequencies for {neuron_type} Neurons')
    plt.legend()
    plt.show()


In [ ]:
#combined line plots 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load necessary columns from the CSV
data = pd.read_csv("/Users/rohan/Documents/Research/Extended_Metrics.csv", usecols=['DIV', 'NeuronType', 'ISIString'])

# Filter data for DIV == 20
data = data[data['DIV'] == 20]

# Function to remove commas, semicolons, and convert string to a list of floats
def process_isi_string(isi_string):
    try:
        # Replace commas and semicolons with spaces
        clean_string = isi_string.replace(',', ' ').replace(';', ' ')
        # Split the cleaned string into parts and convert to float
        return [float(x) for x in clean_string.split()]
    except ValueError as e:
        print(f"Could not convert string to float: '{isi_string}' - Error: {e}")
        return None

# Apply the function to the ISIString column
data['ProcessedISI'] = data['ISIString'].apply(process_isi_string)

# Remove rows where conversion failed
data = data[data['ProcessedISI'].notna()]

# Calculate instantaneous frequencies (1/ISI)
data['InstantFreq'] = data['ProcessedISI'].apply(lambda x: [1/i for i in x if i != 0])

# Define bins for histogram - increased number of bins for finer resolution
bins = np.linspace(0, 50, 201)  # Increase to 200 bins

# Function to compute histogram for each row
def compute_histogram(instant_freq):
    hist, _ = np.histogram(instant_freq, bins=bins)
    return hist

# Compute histograms for each row
data['Histogram'] = data['InstantFreq'].apply(compute_histogram)

# Aggregate histograms by neuron type
grouped_histograms = data.groupby('NeuronType')['Histogram'].agg(list)

# Plotting
plt.figure(figsize=(14, 8))
bin_centers = 0.5 * (bins[:-1] + bins[1:])

for neuron_type, histograms in grouped_histograms.items():
    # Stack histograms into a 2D array
    histograms_stack = np.vstack(histograms)
    mean_hist = np.mean(histograms_stack, axis=0)
    std_hist = np.std(histograms_stack, axis=0)

    # Plot mean histogram with error bars
    plt.errorbar(bin_centers, mean_hist, yerr=std_hist, fmt='-o', label=f'{neuron_type} Neurons', capsize=5)

plt.xlabel('Instantaneous Frequency (Hz)')
plt.ylabel('Average Frequency Count')
plt.title('Average Histogram of Instantaneous Frequencies for WT and HET Neurons')
plt.legend()
plt.grid(True)
plt.ylim(0, np.max([h.max() for h in histograms_stack]) + 5)  # Set Y-axis limits to max histogram value +5
plt.show()


In [ ]:
#using log scale and histogram 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load necessary columns from the CSV
data = pd.read_csv("/Users/rohan/Documents/Research/Extended_Metrics.csv", usecols=['DIV', 'NeuronType', 'ISIString'])

# Filter data for DIV == 20
data = data[data['DIV'] == 20]

# Function to remove commas, semicolons, and convert string to a list of floats
def process_isi_string(isi_string):
    try:
        # Replace commas and semicolons with spaces
        clean_string = isi_string.replace(',', ' ').replace(';', ' ')
        # Split the cleaned string into parts and convert to float
        return [float(x) for x in clean_string.split()]
    except ValueError as e:
        print(f"Could not convert string to float: '{isi_string}' - Error: {e}")
        return None

# Apply the function to the ISIString column
data['ProcessedISI'] = data['ISIString'].apply(process_isi_string)

# Remove rows where conversion failed
data = data[data['ProcessedISI'].notna()]

# Calculate instantaneous frequencies (1/ISI)
data['InstantFreq'] = data['ProcessedISI'].apply(lambda x: [1/i for i in x if i != 0])

# Define logarithmic bins for the histogram
bin_edges = np.logspace(np.log10(0.1), np.log10(300), 50)  # Start from 0.1 to avoid log(0)

# Plotting histograms for WT and HET neuron types in a single graph
plt.figure(figsize=(14, 6))
colors = {'WT': 'blue', 'HET': 'orange'}

# Plot histograms for both neuron types
for neuron_type, group in data.groupby('NeuronType'):
    all_freqs = [freq for freq in np.concatenate(group['InstantFreq'].tolist()) if freq > 0]
    plt.hist(all_freqs, bins=bin_edges, color=colors[neuron_type], alpha=0.7, label=f'{neuron_type} Neurons', log=True)

plt.title('Combined Histograms of Instantaneous Frequencies for WT and HET Neurons')
plt.xlabel('Instantaneous Frequency (Hz) [Log Scale]')
plt.ylabel('Frequency Count')
plt.xscale('log')
plt.xlim(0.1, 300)  # Set x-axis limits to show from 0.1 to 300 Hz
plt.legend()
plt.grid(True, which="both", ls="--")
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load necessary columns from the CSV
data = pd.read_csv("/Users/rohan/Documents/Research/forISI.csv", usecols=['DIV', 'NeuronType', 'ISIString'], engine='python')

# Filter data for DIV == 20
data = data[data['DIV'] == 20]

# Function to remove commas, semicolons, and convert string to a list of floats
def process_isi_string(isi_string):
    try:
        # Replace commas and semicolons with spaces
        clean_string = isi_string.replace(',', ' ').replace(';', ' ')
        # Split the cleaned string into parts and convert to float
        return [float(x) for x in clean_string.split()]
    except ValueError as e:
        print(f"Could not convert string to float: '{isi_string}' - Error: {e}")
        return None

# Apply the function to the ISIString column
data['ProcessedISI'] = data['ISIString'].apply(process_isi_string)

# Remove rows where conversion failed
data = data[data['ProcessedISI'].notna()]

# Calculate instantaneous frequencies (1/ISI)
data['InstantFreq'] = data['ProcessedISI'].apply(lambda x: [1/i for i in x if i != 0])

# Define logarithmic bins for the histogram
bin_edges = np.logspace(np.log10(0.1), np.log10(300), 50)  # Start from 0.1 to avoid log(0)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

# Function to compute histograms and return normalized count
def compute_histogram(instant_freq):
    hist, _ = np.histogram(instant_freq, bins=bin_edges, density=True)
    return hist

# Compute histograms for each entry
data['Histogram'] = data['InstantFreq'].apply(compute_histogram)

# Group data by NeuronType and aggregate histograms
grouped_histograms = data.groupby('NeuronType')['Histogram'].agg(list)

# Plotting
plt.figure(figsize=(14, 7))

for neuron_type, histograms in grouped_histograms.items():
    histograms_stack = np.vstack(histograms)
    mean_hist = np.mean(histograms_stack, axis=0)
    sem_hist = np.std(histograms_stack, axis=0) / np.sqrt(histograms_stack.shape[0])  # Calculating SEM

    plt.errorbar(bin_centers, mean_hist, yerr=sem_hist, fmt='-o', label=f'{neuron_type} Neurons', capsize=5)

plt.xlabel('Instantaneous Frequency (Hz) [Log Scale]')
plt.ylabel('Normalized Frequency Count')
plt.xscale('log')
plt.legend()
plt.grid(True, which="both", ls="--")
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import csv
import sys

# Increase the maximum field size allowed
csv.field_size_limit(sys.maxsize)

# Load necessary columns from the CSV
data_unfiltered = pd.read_csv("/Users/rohan/Documents/Research/Extended_Metrics.csv", usecols=['DIV', 'NeuronType', 'ISIString'], engine='python')
print("Unique DIV values:", data_unfiltered['DIV'].unique())

# Filter data for an existing DIV value
data = data_unfiltered[data_unfiltered['DIV'] == 24].copy()
print("Data count after filtering for DIV == 24:", data.shape[0])

if not data.empty:
    def process_isi_string(isi_string):
        try:
            clean_string = isi_string.replace(',', ' ').replace(';', ' ')
            return [float(x) for x in clean_string.split()]
        except ValueError as e:
            print(f"Could not convert string to float: '{isi_string}' - Error: {e}")
            return None

    data['ProcessedISI'] = data['ISIString'].apply(process_isi_string)
    data = data[data['ProcessedISI'].notna()]

    data['InstantFreq'] = data['ProcessedISI'].apply(lambda x: [1/i for i in x if i != 0])

    bin_edges = np.logspace(np.log10(0.1), np.log10(300), 50)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

    def compute_histogram(instant_freq):
        hist, _ = np.histogram(instant_freq, bins=bin_edges, density=True)
        return hist

    data['Histogram'] = data['InstantFreq'].apply(compute_histogram)
    grouped_histograms = data.groupby('NeuronType')['Histogram'].agg(list)

    plt.figure(figsize=(14, 7))
    colors = {'WT cortex': 'blue', 'HET cortex': 'red'}  # Define specific colors for neuron types
    for neuron_type, histograms in grouped_histograms.items():
        histograms_stack = np.vstack(histograms)
        mean_hist = np.mean(histograms_stack, axis=0)
        sem_hist = np.std(histograms_stack, axis=0) / np.sqrt(histograms_stack.shape[0])

        current_color = colors.get(neuron_type, 'black')  # Default to black if no specific color
        plt.errorbar(bin_centers, mean_hist, yerr=sem_hist, fmt='-o', color=current_color, label=f'{neuron_type} Neurons', capsize=5)
        plt.fill_between(bin_centers, 0, mean_hist, color=current_color, alpha=0.1)  # Fill from 0 to line

    plt.xlabel('Instantaneous Frequency (Hz) [Log Scale]')
    plt.ylabel('Normalized Frequency Count')
    plt.xscale('log')
    plt.legend()
    plt.grid(False)  # Disable the grid
    plt.show()
else:
    print("No data available after filtering.")

In [ ]:
#new code without removing the ';' 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load necessary columns from the CSV
data = pd.read_csv("/Users/rohan/Documents/Research/forISI.csv", usecols=['DIV', 'NeuronType', 'ISIString'])

# Filter data for DIV == 20
data = data[data['DIV'] == 20]

# Function to process ISI strings, taking channels into account
def process_isi_string(isi_string):
    channels = isi_string.split(';')  # Split channels by ';'
    freq_list = []
    for channel in channels:
        try:
            # Convert each part to float after removing commas
            isis = [float(x) for x in channel.replace(',', ' ').split()]
            # Calculate instantaneous frequencies for non-zero ISIs
            freq_list.extend([1/i for i in isis if i != 0])
        except ValueError as e:
            print(f"Error processing channel: '{channel}' - Error: {e}")
    return freq_list

# Apply the function to the ISIString column
data['InstantFreq'] = data['ISIString'].apply(process_isi_string)

# Remove rows where conversion might have failed completely (if any)
data = data[data['InstantFreq'].apply(len) > 0]

# Function to perform binning with finer bins
def bin_instant_freq(data, bins=100):  # Set bins to 100 for finer resolution
    # Concatenate all frequencies into a single list
    all_freqs = np.concatenate(data['InstantFreq'].tolist())
    # Calculate histogram
    hist, bin_edges = np.histogram(all_freqs, bins=bins)
    # Calculate bin centers
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    return bin_centers, hist

# Data for 'WT' and 'HET'
data_wt = data[data['NeuronType'] == 'WT']
data_het = data[data['NeuronType'] == 'HET']

# Bin the frequencies with finer resolution
bin_centers_wt, hist_wt = bin_instant_freq(data_wt)
bin_centers_het, hist_het = bin_instant_freq(data_het)

# Plotting separate graphs for WT and HET with finer bins
plt.figure(figsize=(14, 6))
plt.subplot(1, 2, 1)  # WT histogram
plt.bar(bin_centers_wt, hist_wt, width=np.diff(bin_centers_wt)[0], alpha=0.5, color='blue')
plt.title('WT Neurons - Instantaneous Frequencies')
plt.xlabel('Instantaneous Frequency (Hz)')
plt.ylabel('Count')

plt.subplot(1, 2, 2)  # HET histogram
plt.bar(bin_centers_het, hist_het, width=np.diff(bin_centers_het)[0], alpha=0.5, color='red')
plt.title('HET Neurons - Instantaneous Frequencies')
plt.xlabel('Instantaneous Frequency (Hz)')
plt.ylabel('Count')

plt.suptitle('Detailed Histogram of Instantaneous Frequencies for WT and HET Neurons by Channel')
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


In [ ]:

isi_string=isi_string[:].split()
isi_list = [float(x) for x in isi_string]

# Print the resulting list
print(isi_list)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from ast import literal_eval

# Try reading the CSV with adjusted parameters
try:
    data = pd.read_csv("/Users/rohan/Documents/Research/forISI.csv", quoting=3, on_bad_lines='skip')
except Exception as e:
    print(f"Failed to read CSV: {e}")

# Ensure data is loaded
if 'data' in locals():
    # Filter data for DIV == 20
    data = data[data['DIV'] == 20]

    # Function to calculate instantaneous frequencies from ISI strings
    def calculate_instantaneous_frequencies(isi_string):
        try:
            # Ensure the string is formatted as a list if not already
            if isi_string.startswith('[') and isi_string.endswith(']'):
                isi_list = literal_eval(isi_string)  # Convert string to list
            else:
                isi_list = literal_eval(f"[{isi_string}]")  # Add brackets to form a valid list
            return [1 / isi for isi in isi_list if isi != 0]
        except Exception as e:
            print(f"Error processing ISIString: {isi_string} | Error: {e}")
        return []

    # Apply the function and create a new column for instantaneous frequencies
    data['Instant_Freq'] = data['ISIString'].apply(calculate_instantaneous_frequencies)

In [ ]:
# Function to calculate instantaneous frequencies from ISI strings
def calculate_instantaneous_frequencies(isi_string):
    isi_list = literal_eval(isi_string)
    if isi_list:
        return [1 / isi for isi in isi_list if isi != 0]
    return []

# Apply the function and create a new column for instantaneous frequencies
data['Instant_Freq'] = data['ISIString'].apply(calculate_instantaneous_frequencies)

# Separate data for 'WT' and 'HET'
data_wt = data[data['NeuronType'] == 'WT']
data_het = data[data['NeuronType'] == 'HET']